# 03 · Hyperspectral roof-material screening (Tier 2)

**Run this only after `02_main_analysis.ipynb` has completed successfully.**
Tier 1 is a complete submission on its own; this adds the hyperspectral bonus.

---

### Where we stand

`00_recon.ipynb` checked the four Tanager `urban` scene IDs the organizers use and
found **none over the UAE**. But the full-catalogue scan **crashed** with a JSON
decode error — it never actually looked. So the question is still open, and this
notebook settles it properly.

### What we do, in order

1. **Walk the Tanager STAC catalogue correctly** (section 1). If a UAE urban scene
   exists, we use it and stop here.
2. **If none exists — use the Riyadh scene as a declared analogue** (section 2).
   One known scene sits at centroid (46.86 °E, 24.58 °N): Saudi Arabia, essentially
   the same latitude as Abu Dhabi (24.4 °N), the same hyper-arid Gulf climate, and
   the same regional construction materials. Demonstrating the method there and
   saying so plainly is honest and defensible. Pretending it is Abu Dhabi is not.
3. **If neither works — document the gap and stop.** A clean, explained negative
   scores better under *validity* than a fabricated positive. The programme's own
   guidance lists inventing coverage among the mistakes that sink a PoC.

### Why hyperspectral adds something real here

Albedo tells you how *bright* a roof is. It cannot tell you what the roof is *made
of* — and material decides which retrofit is appropriate and what it costs. Coating
works on concrete and bitumen; a corroded metal deck may need replacement instead.
Six broad Sentinel-2 bands cannot separate those. 426 narrow bands can.


In [ ]:
%pip install -q requests h5py numpy matplotlib shapely scikit-learn

In [ ]:
import os, sys, json, requests, subprocess
import numpy as np, matplotlib.pyplot as plt
from shapely.geometry import box

REPO_DIR = "coolroof-mussafah"
if not (os.path.isdir("../results") and os.path.isdir("../data")):
    if os.path.isdir(f"/content/{REPO_DIR}/notebooks"):
        os.chdir(f"/content/{REPO_DIR}/notebooks")
    elif "google.colab" in sys.modules:
        os.chdir("/content")
        if not os.path.isdir(REPO_DIR):
            subprocess.run(["git","clone","-q",
                "https://github.com/RISKYXBT/coolroof-mussafah.git", REPO_DIR])
        os.chdir(f"/content/{REPO_DIR}/notebooks")
os.makedirs("../results", exist_ok=True); os.makedirs("../data/sample_input", exist_ok=True)
RESULTS, SAMPLE = "../results", "../data/sample_input"

BASE    = "https://www.planet.com/data/stac/tanager-core-imagery"
UAE_BOX = box(51.0, 22.5, 56.5, 26.5)
print("working directory:", os.getcwd())

## 1 · Walk the Tanager catalogue properly

The recon notebook tried `.../urban/catalog.json` directly and got HTML back instead
of JSON. Here we start from the root catalogue and follow `child` links the way a
STAC client should, trying several entry points.

In [ ]:
def get_json(url, timeout=60):
    try:
        r = requests.get(url, timeout=timeout)
        if r.status_code != 200:
            return None, f"HTTP {r.status_code}"
        return r.json(), None
    except Exception as e:
        return None, f"{type(e).__name__}: {e}"

# Try several plausible entry points for the root catalogue.
roots = [f"{BASE}/catalog.json", f"{BASE}/", BASE]
root = None
for u in roots:
    j, err = get_json(u)
    if j and isinstance(j, dict) and ("links" in j or "type" in j):
        root, root_url = j, u
        print(f"Root catalogue OK at {u}")
        print(f"   type={j.get('type')} id={j.get('id')}")
        break
    print(f"   {u} -> {err or 'unexpected payload'}")

if root:
    children = [l for l in root.get("links", []) if l.get("rel") in ("child","collection")]
    print(f"\n{len(children)} child collections:")
    for l in children[:30]:
        print("   ", l.get("title") or l.get("href"))
else:
    print("\nCould not reach the root catalogue from this network.")

In [ ]:
def resolve(href, parent_url):
    if href.startswith("http"):
        return href
    base = parent_url.rsplit("/", 1)[0]
    return f"{base}/{href.lstrip('./')}"

uae_hits, checked = [], 0
if root:
    targets = [l for l in root.get("links", [])
               if l.get("rel") in ("child","collection")
               and "urban" in (str(l.get("href","")) + str(l.get("title",""))).lower()]
    if not targets:
        targets = [l for l in root.get("links", []) if l.get("rel") in ("child","collection")]

    for l in targets:
        curl = resolve(l["href"], root_url)
        coll, err = get_json(curl)
        if not coll:
            print(f"collection {curl} -> {err}")
            continue
        items = [x for x in coll.get("links", []) if x.get("rel") == "item"]
        print(f"\n{coll.get('id', curl)}: {len(items)} items")
        for x in items:
            iurl = resolve(x["href"], curl)
            it, e2 = get_json(iurl, timeout=30)
            checked += 1
            if not it or "bbox" not in it:
                continue
            b = it["bbox"]
            if box(b[0], b[1], b[2], b[3]).intersects(UAE_BOX):
                uae_hits.append({"id": it["id"],
                                 "date": it["properties"].get("datetime","")[:10],
                                 "bbox": b, "collection": coll.get("id"),
                                 "assets": sorted(it.get("assets", {}))})
                print(f"   *** UAE HIT: {it['id']}  {b}")
            if checked % 25 == 0:
                print(f"   ...{checked} items checked, {len(uae_hits)} hits")

print(f"\n==== {checked} items checked, {len(uae_hits)} intersect the UAE ====")
for h in uae_hits:
    print(json.dumps(h, indent=2))

## 2 · Decide the path

Run this. It prints which of the three routes applies and sets `MODE` accordingly.

In [ ]:
RIYADH_ID = "20250407_035509_25_4001"   # centroid ~ (46.86 E, 24.58 N)

if uae_hits:
    MODE, SCENE_ID, COLLECTION = "UAE", uae_hits[0]["id"], uae_hits[0]["collection"]
    print(f"MODE = UAE. Using {SCENE_ID} - a genuine Tanager scene over the UAE.")
else:
    item, err = get_json(f"{BASE}/urban/{RIYADH_ID}/{RIYADH_ID}.json")
    if item:
        b = item["bbox"]
        MODE, SCENE_ID, COLLECTION = "ANALOGUE", RIYADH_ID, "urban"
        print("MODE = ANALOGUE.")
        print(f"No Tanager urban scene covers the UAE. Using {RIYADH_ID} over Saudi")
        print(f"Arabia as a DECLARED analogue.  bbox={[round(v,3) for v in b]}")
        print(f"  latitude {b[1]:.2f}-{b[3]:.2f} N  vs  Mussafah 24.29-24.41 N")
        print("  Same hyper-arid Gulf climate and regional construction materials.")
        print("  This MUST be labelled as an analogue in the README and on the slide.")
    else:
        MODE = "NONE"
        print(f"MODE = NONE. Riyadh scene unreachable ({err}).")
        print("Document the gap in README section 9 and stop. Do not invent coverage.")

## 3 · Download the cube and read its spectra

Uses the same HDF5 layout and quality-mask logic as the organizers' notebook 02, so
a reviewer familiar with the starter material recognises it immediately.

In [ ]:
if MODE in ("UAE", "ANALOGUE"):
    item, _ = get_json(f"{BASE}/{COLLECTION}/{SCENE_ID}/{SCENE_ID}.json")
    sr_key = "ortho_sr_hdf5" if "ortho_sr_hdf5" in item["assets"] else "basic_sr_hdf5"
    sr_url = item["assets"][sr_key]["href"]
    local  = os.path.join(SAMPLE, os.path.basename(sr_url))

    if not os.path.exists(local):
        print("Downloading hyperspectral cube - several minutes, go get coffee...")
        with requests.get(sr_url, stream=True, timeout=1200) as r:
            r.raise_for_status()
            total = 0
            with open(local, "wb") as fh:
                for chunk in r.iter_content(chunk_size=4*1024*1024):
                    fh.write(chunk); total += len(chunk)
                    if total % (80*1024*1024) < 4*1024*1024:
                        print(f"   {total/1e6:.0f} MB")
    print("cube:", local, f"{os.path.getsize(local)/1e6:.0f} MB")

    bands_meta = item["assets"][sr_key].get("bands", [])
    wl = np.array([b["eo:center_wavelength"] for b in bands_meta
                   if "eo:center_wavelength" in b]) * 1000.0
    print(f"{len(wl)} spectral bands, {wl.min():.0f}-{wl.max():.0f} nm")
else:
    print("Skipped - MODE is NONE.")

In [ ]:
if MODE in ("UAE", "ANALOGUE"):
    import h5py
    # Drop the water-vapour absorption troughs and the noisy SWIR detector edge.
    # Same treatment as the organizers' notebooks - these bands carry no surface
    # signal, only atmosphere and sensor noise.
    def bad(w): return (1350 <= w <= 1450) or (1800 <= w <= 1950) or w > 2450
    good = [i for i, w in enumerate(wl) if not bad(w)]
    wls  = wl[good]
    print(f"Keeping {len(good)} of {len(wl)} bands after removing water-vapour "
          f"troughs and the SWIR edge")

    root_h5 = "HDFEOS/GRIDS/HYP/Data Fields"
    with h5py.File(local, "r") as f:
        cloud  = f[f"{root_h5}/beta_cloud_mask"][:]
        nodata = f[f"{root_h5}/nodata_pixels"][:]
        valid  = (cloud == 0) & (nodata == 0)
        cube   = f[f"{root_h5}/surface_reflectance"][good, :, :].astype("float32")
    cube[:, ~valid] = np.nan
    cube[cube < 0]  = np.nan
    print(f"cube (bands, y, x) = {cube.shape}, {valid.mean()*100:.1f}% valid pixels")

    # Quick RGB-ish view to orient yourself before picking endmembers.
    def nearest(t): return int(np.argmin(np.abs(wls - t)))
    rgb = np.dstack([cube[nearest(660)], cube[nearest(550)], cube[nearest(470)]])
    rgb = np.clip(rgb / np.nanpercentile(rgb, 98), 0, 1)
    plt.figure(figsize=(9, 8)); plt.imshow(rgb); plt.axis("off")
    plt.title(f"Tanager {SCENE_ID} - true colour ({MODE})")
    plt.savefig(f"{RESULTS}/06_tanager_rgb.png", bbox_inches="tight"); plt.show()
    print("\nUse this image to pick endmember pixels (row, col) for the next cell.")

## 4 · Unsupervised material clustering

Rather than hand-picking endmember pixels — which is subjective and hard for a
reviewer to reproduce — we cluster the spectra with k-means. Every pixel's full
spectral signature goes in; the clusters come out of the data.

We then plot each cluster's mean spectrum. Material identity is read off those
curves, and we **label them as spectral classes, not confirmed materials**, because
we have no field verification. That distinction matters under *validity*.

In [ ]:
if MODE in ("UAE", "ANALOGUE"):
    from sklearn.cluster import KMeans
    np.random.seed(42)

    nb_, ny, nx = cube.shape
    flat = cube.reshape(nb_, -1).T                 # (pixels, bands)
    ok   = np.isfinite(flat).all(axis=1)
    print(f"{ok.sum():,} fully valid pixels of {flat.shape[0]:,}")

    # Normalise each spectrum to unit length: clusters then key on spectral SHAPE
    # (material) rather than overall brightness (illumination and slope).
    Xs = flat[ok]
    Xs = Xs / (np.linalg.norm(Xs, axis=1, keepdims=True) + 1e-9)

    K = 6
    sub = Xs[np.random.choice(len(Xs), size=min(40000, len(Xs)), replace=False)]
    km  = KMeans(n_clusters=K, n_init=10, random_state=42).fit(sub)
    lab = km.predict(Xs)

    cls = np.full(flat.shape[0], np.nan); cls[ok] = lab
    cls = cls.reshape(ny, nx)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6.5))
    im = axes[0].imshow(cls, cmap="tab10", vmin=0, vmax=K-1)
    axes[0].set_title(f"Spectral classes, k-means k={K}"); axes[0].axis("off")
    plt.colorbar(im, ax=axes[0], ticks=range(K), fraction=0.046)

    for k in range(K):
        m = Xs[lab == k].mean(axis=0)
        axes[1].plot(wls, m, lw=1.4, label=f"class {k}  ({(lab==k).mean()*100:.1f}%)")
    axes[1].set_xlabel("Wavelength (nm)")
    axes[1].set_ylabel("Normalised reflectance (unit-norm spectrum)")
    axes[1].set_title("Mean spectrum per class")
    axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{RESULTS}/07_material_classes.png", bbox_inches="tight"); plt.show()

    print("\nReading the curves - what to look for:")
    print("  bare sand / carbonate : steady rise through VNIR, dip near 2200 nm")
    print("  concrete              : flat-to-rising, moderate brightness, weak 2200 nm")
    print("  bitumen / asphalt     : low and flat across the whole range")
    print("  metal roofing         : bright, often a distinctly flat SWIR")
    print("  vegetation            : red edge near 700 nm, water dips 970/1200 nm")
    print("\nLabel these as SPECTRAL CLASSES in the write-up. We have no field")
    print("verification, so calling them confirmed materials would overclaim.")

In [ ]:
if MODE in ("UAE", "ANALOGUE"):
    out = {
        "mode": MODE,
        "scene_id": SCENE_ID,
        "collection": COLLECTION,
        "bbox": item["bbox"],
        "datetime": item["properties"].get("datetime"),
        "bands_total": int(len(wl)),
        "bands_used": int(len(good)),
        "bands_removed": "water-vapour 1350-1450 and 1800-1950 nm, SWIR edge >2450 nm",
        "valid_pixel_pct": round(float(valid.mean()*100), 1),
        "method": "unit-norm spectra, k-means k=6, seed 42",
        "caveat": ("Spectral classes, not field-verified materials." +
                   (" Scene is over Saudi Arabia and is used as a DECLARED climatic "
                    "and construction analogue for Abu Dhabi; no Tanager urban scene "
                    "covers the UAE." if MODE == "ANALOGUE" else "")),
        "licence": "CC BY 4.0 (c) Planet Labs PBC",
    }
    with open(f"{RESULTS}/hyperspectral_summary.json", "w") as f:
        json.dump(out, f, indent=2)
    print(json.dumps(out, indent=2))
else:
    with open(f"{RESULTS}/hyperspectral_summary.json", "w") as f:
        json.dump({"mode": "NONE",
                   "finding": ("No Planet Tanager urban scene intersects the UAE, and "
                               "the Riyadh analogue scene was unreachable. The "
                               "hyperspectral layer is specified but not executed."),
                   "checked_items": checked}, f, indent=2)
    print("Wrote the negative finding. Report it in README section 9 as-is.")

---

## What to put in the README

**If MODE was UAE** — report it as a genuine result: scene ID, date, the classes
found, and the caveat that they are spectral classes rather than verified materials.

**If MODE was ANALOGUE** — say this, in these words or close to them:

> No Planet Tanager `urban` scene intersects the UAE within the PoC data window
> (verified by walking the open STAC catalogue; see `00_recon.ipynb` and
> `03_hyperspectral.ipynb`). We therefore demonstrate the roof-material screening
> method on Tanager scene `<id>` over Saudi Arabia at 24.6 °N — the same latitude,
> hyper-arid climate and regional construction practice as Mussafah at 24.4 °N — and
> label it explicitly as an analogue. The method transfers to Abu Dhabi the moment a
> scene is acquired; the result shown is not Abu Dhabi.

That paragraph is worth more than a quietly mislabelled map. It shows you checked,
you found a limitation, and you handled it in the open — which is exactly what the
*validity* criterion rewards.

**If MODE was NONE** — state the gap plainly and move on. Tier 1 stands by itself.
